# Cluster table ready for word

## install required libraries

In [1]:
!pip install python-docx
!pip install neurosynth


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Main code

## 2 tables (1 main 1 covar)

In [8]:
import os
import re
import numpy as np
import nibabel as nib
from pathlib import Path
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# Base directory - updated for new Linux path
base_dir_str = "/home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer"
results_dir = Path(base_dir_str) / "results"
effect_maps_base = Path(base_dir_str)

# Function to set cell borders
def set_cell_border(cell, **kwargs):
    """Set cell borders"""
    tc = cell._element
    tcPr = tc.get_or_add_tcPr()

    # Remove existing borders
    tcBorders = tcPr.find(qn('w:tcBorders'))
    if tcBorders is not None:
        tcPr.remove(tcBorders)
    
    # Add new borders
    tcBorders = OxmlElement('w:tcBorders')
    
    for edge in ('top', 'bottom', 'start', 'end'):
        if edge in kwargs:
            border = OxmlElement(f'w:{edge}')
            for key, value in kwargs[edge].items():
                border.set(qn(f'w:{key}'), str(value))
            tcBorders.append(border)
    
    tcPr.append(tcBorders)

# Function to make table row repeat as header
def set_repeat_table_header(row):
    """Set table row to repeat as header on each page"""
    tr = row._element
    trPr = tr.get_or_add_trPr()
    tblHeader = OxmlElement('w:tblHeader')
    tblHeader.set(qn('w:val'), "true")
    trPr.append(tblHeader)

# Function to extract information from filename
def parse_filename(filepath):
    """Parse the filename to extract model, second level contrast, and first level contrast"""
    filename = filepath.name
    parent_dir = filepath.parent.parent.name
    
    # Determine if it's a TWL model - includes _twl_24 or _covar folders
    is_twl = "_twl_24" in parent_dir or parent_dir.endswith("_covar")
    
    # Check if this is a _covar folder (simplified naming)
    is_covar_folder = parent_dir.endswith('_covar')
    
    if is_covar_folder:
        # For _covar folders: simpler file naming without _p001_ACF_residuals_
        # Files are like: sess1_x_twl24_clustered_Brainnetome...
        base_name = filename.replace('_clustered_Brainnetome_1.0_whereami_report.txt', '')
        
        # Extended session map for _covar folders
        session_map = {
            'sess1_x_twl24': 'pre-surgery',
            'sess1': 'pre-surgery',
            'sess2vs1_x_twl24': '4 months post-surgery > pre-surgery',
            'sess2vs1': '4 months post-surgery > pre-surgery',
            'sess3vs1_x_twl24': '12 months post-surgery > pre-surgery',
            'sess3vs1': '12 months post-surgery > pre-surgery',
            'sess4vs1_x_twl24': '24 months post-surgery > pre-surgery',
            'sess4vs1': '24 months post-surgery > pre-surgery',
            'sess2_x_twl24': '4 months post-surgery',
            'sess2': '4 months post-surgery',
            'sess3_x_twl24': '12 months post-surgery',
            'sess3': '12 months post-surgery',
            'sess4_x_twl24': '24 months post-surgery',
            'sess4': '24 months post-surgery',
            'bmi_baseline': 'BMI baseline',
            'age_baseline': 'Age baseline',
            'type_chx_3vs1': 'Surgery type (BPD > SG) at 12 months',
            'type_chx_2vs1': 'Surgery type (RYGB > SG) at 4 months',
            'run_3vs1': 'Run 3 > Run 1',
            'run_2vs1': 'Run 2 > Run 1',
            'handedness_LvR': 'Handedness (left > right)',
            'twl_rech_24': 'TWL at 24 months',
        }
        
        # Find matching session
        second_level_contrast = None
        for key, value in session_map.items():
            if base_name == key or base_name.startswith(f"{key}_"):
                if is_twl and '_x_twl24' in key:
                    second_level_contrast = f"{value} X TWL at 24 months"
                else:
                    second_level_contrast = value
                break
    else:
        # Original logic for non-_covar folders
        if is_twl:
            if not any(filename.startswith(key) for key in ['sess1_x_twl24_', 'sess2vs1_x_twl24_', 
                                                             'sess3vs1_x_twl24_', 'sess4vs1_x_twl24_']):
                return None, None, None
        else:
            if any(filename.startswith(key) for key in ['sess1_x_twl24_', 'sess2vs1_x_twl24_', 
                                                         'sess3vs1_x_twl24_', 'sess4vs1_x_twl24_']):
                return None, None, None
        
        session_map = {
            'sess1_x_twl24_': 'pre-surgery',
            'sess1_': 'pre-surgery',
            'sess2vs1_x_twl24_': '4 months post-surgery > pre-surgery',
            'sess2vs1_': '4 months post-surgery > pre-surgery',
            'sess3vs1_x_twl24_': '12 months post-surgery > pre-surgery',
            'sess3vs1_': '12 months post-surgery > pre-surgery',
            'sess4vs1_x_twl24_': '24 months post-surgery > pre-surgery',
            'sess4vs1_': '24 months post-surgery > pre-surgery',
            'bmi_baseline_': 'BMI baseline',
            'age_baseline_': 'Age baseline',
            'type_chx_3vs1_': 'Surgery type (BPD > SG) at 12 months',
            'type_chx_2vs1_': 'Surgery type (RYGB > SG) at 4 months',
            'run_3vs1_': 'Run 3 > Run 1',
            'run_2vs1_': 'Run 2 > Run 1',
            'handedness_lvs_r_': 'Handedness (left > right)',
        }
        
        second_level_contrast = None
        for key, value in session_map.items():
            if filename.startswith(key):
                if is_twl:
                    second_level_contrast = f"{value} X TWL at 24 months"
                else:
                    second_level_contrast = value
                break
    
    # Model description
    if is_twl:
        model = "session*TWL + covariates"
    else:
        model = "session + covariates"
    
    # First level contrast - check parent directory name
    first_level_contrast = None
    if "mod_hi_vs_mod_lo" in parent_dir:
        first_level_contrast = "Valuating high calorie > low calorie"
    elif "view_Hi_vs_view_Lo" in parent_dir or "view_Hi_vs_view_low" in parent_dir:
        first_level_contrast = "Viewing high calorie > low calorie"
    elif "mod_view_Hi_vs_0" in parent_dir or "view_Hi_vs_0" in parent_dir:
        first_level_contrast = "Valuating high calorie"
    elif "mod_view_Lo_vs_0" in parent_dir or "view_Lo_vs_0" in parent_dir:
        first_level_contrast = "Valuating low calorie"
    elif "mod_view_vs_0" in parent_dir or "view_vs_0" in parent_dir:
        first_level_contrast = "Valuating (all conditions)"
    
    return model, second_level_contrast, first_level_contrast

def find_cluster_and_effect_files(report_filepath):
    """Find corresponding cluster and effect size NIfTI files based on report filename"""
    parent_dir = report_filepath.parent.parent.name
    filename = report_filepath.name
    
    # Check if this is a _covar folder (simplified naming structure)
    is_covar_folder = parent_dir.endswith('_covar')
    
    # Extract the base pattern from the report filename
    base_name = filename.replace('_Brainnetome_1.0_whereami_report.txt', '')
    
    # Find cluster file in results directory
    cluster_file = results_dir / parent_dir / f"{base_name}.nii.gz"
    
    # Parse the base_name to extract session and contrast info
    session_part = None
    contrast_part = None
    
    # Extended list of session patterns - ORDER MATTERS! Longer patterns first
    session_patterns = [
        'bmi_baseline_chisq', 'sess1_x_twl24', 'sess2vs1_x_twl24', 'sess3vs1_x_twl24', 'sess4vs1_x_twl24',
        'handedness_chisq', 'sess_chisq', 'handedness_LvR', 'twl_rech_24',
        'type_chx_3vs1', 'type_chx_2vs1',
        'run_3vs1', 'run_2vs1',
        'handedness_lvs_r',
        'bmi_baseline', 'age_baseline',
        'sess2vs1', 'sess3vs1', 'sess4vs1', 'sess1', 'sess2', 'sess3', 'sess4',
        'sess2_x_twl24', 'sess3_x_twl24', 'sess4_x_twl24'
    ]
    
    if is_covar_folder:
        # Simplified structure: just extract session pattern from base_name
        base_name_clean = base_name.replace('_clustered', '')
        for key in session_patterns:
            if base_name_clean == key or base_name_clean.startswith(f"{key}_"):
                session_part = key
                # CRITICAL FIX: For _covar folders, use parent_dir as contrast_part
                contrast_part = parent_dir
                break
    else:
        # Original structure with p001_ACF_residuals
        for key in session_patterns:
            if base_name.startswith(key):
                session_part = key
                # Extract everything AFTER "pattern_clustered_p001_ACF_residuals_"
                pattern = f'{key}_clustered_p001_ACF_residuals_'
                if pattern in base_name:
                    remaining = base_name.split(pattern, 1)[1]
                    contrast_part = remaining
                    break
                break
    
    effect_file = None
    z_file = None
    
    if session_part and contrast_part:
        # Look for effect_maps directory
        effect_maps_dir_name = f"effect_maps_{contrast_part}"
        effect_maps_dir = effect_maps_base / effect_maps_dir_name
        
        # If not found, try alternative naming patterns
        if not effect_maps_dir.exists():
            # Try replacing _covariates with _covar
            if "_covariates" in contrast_part:
                alternate_contrast = contrast_part.replace("_covariates", "_covar")
                effect_maps_dir = effect_maps_base / f"effect_maps_{alternate_contrast}"
            
            # If still not found, try adding _qc_final_covar suffix if not present
            if not effect_maps_dir.exists() and not contrast_part.endswith("_qc_final_covar"):
                effect_maps_dir = effect_maps_base / f"effect_maps_{contrast_part}_qc_final_covar"
        
        if not effect_maps_dir.exists():
            print(f"    ! Effect maps directory not found: {effect_maps_dir_name}")
            # Try to find directories that might match
            try:
                matching_dirs = list(effect_maps_base.glob(f"effect_maps_*{parent_dir}*"))
                if matching_dirs:
                    print(f"    ! Found similar directories: {[d.name for d in matching_dirs[:2]]}")
            except:
                pass
            return cluster_file, None, None
        
        # Get the actual file prefix - it's the directory name without 'effect_maps_'
        file_prefix = effect_maps_dir.name.replace('effect_maps_', '')
        
        # Convert session format if needed - map to actual file names
        session_suffix_map = {
            'sess2vs1': 'sess2_vs_sess1',
            'sess3vs1': 'sess3_vs_sess1',
            'sess4vs1': 'sess4_vs_sess1',
            'sess2vs1_x_twl24': 'sess2_vs_sess1_x_twl24',
            'sess3vs1_x_twl24': 'sess3_vs_sess1_x_twl24',
            'sess4vs1_x_twl24': 'sess4_vs_sess1_x_twl24',
            'bmi_baseline': 'bmi_baseline',
            'age_baseline': 'age_baseline',
            'bmi_baseline_chisq': 'bmi_baseline_Chisq',
            'type_chx_3vs1': 'type_chx_3_vs_1',
            'type_chx_2vs1': 'type_chx_2_vs_1',
            'run_3vs1': 'run_3_vs_1',
            'run_2vs1': 'run_2_vs_1',
            'handedness_lvs_r': 'handedness_left_vs_right',
            'handedness_LvR': 'handedness_left_vs_right',
            'twl_rech_24': 'twl_rech_24',
            'sess1': 'sess1',
            'sess2': 'sess2',
            'sess3': 'sess3',
            'sess4': 'sess4',
            'sess1_x_twl24': 'sess1_x_twl24',
            'sess2_x_twl24': 'sess2_x_twl24',
            'sess3_x_twl24': 'sess3_x_twl24',
            'sess4_x_twl24': 'sess4_x_twl24',
        }
        
        session_suffix = session_suffix_map.get(session_part, session_part)
        
        # Construct file names: file_prefix_session_suffix
        effect_base = f"{file_prefix}_{session_suffix}"
        z_base = f"{effect_base}_Z"
        
        # Try to find the files
        potential_effect = effect_maps_dir / f"{effect_base}.nii.gz"
        potential_z = effect_maps_dir / f"{z_base}.nii.gz"
        
        if potential_effect.exists():
            effect_file = potential_effect
        else:
            print(f"    ! Effect file not found: {potential_effect}")
            
        if potential_z.exists():
            z_file = potential_z
        else:
            print(f"    ! Z file not found: {potential_z}")
    else:
        if not session_part:
            print(f"    ! Could not identify session pattern in: {base_name}")
        if not contrast_part:
            print(f"    ! Could not extract contrast part from: {base_name}")

    return cluster_file, effect_file, z_file

def calculate_cluster_stats(cluster_file, effect_file, z_file, cluster_id):
    """Calculate Peak Z, Peak MNI coordinates, and Mean ES for a cluster"""
    try:
        # Check if cluster file exists
        if not cluster_file.exists():
            print(f"    ! Cluster file not found: {cluster_file}")
            return None, None, None
            
        # Load cluster mask
        cluster_img = nib.load(str(cluster_file))
        cluster_data = cluster_img.get_fdata()
        affine = cluster_img.affine
        
        # Create mask for this specific cluster
        cluster_mask = (cluster_data == int(cluster_id))
        
        if not np.any(cluster_mask):
            return None, None, None
        
        # Load Z-score map
        peak_z = None
        peak_mni = None
        if z_file and z_file.exists():
            z_img = nib.load(str(z_file))
            z_data = z_img.get_fdata()
            
            # Get Z values within cluster
            z_vals = z_data[cluster_mask]
            if len(z_vals) > 0:
                peak_z = np.max(np.abs(z_vals))
                
                # Find peak coordinates
                abs_z = np.abs(z_data)
                masked_z = abs_z * cluster_mask
                peak_idx = np.unravel_index(np.argmax(masked_z), z_data.shape)
                
                # Convert to MNI coordinates
                mni_coords = nib.affines.apply_affine(affine, peak_idx)
                peak_mni = f"{mni_coords[0]:.0f}, {mni_coords[1]:.0f}, {mni_coords[2]:.0f}"
        
        # Load effect size map
        mean_es = None
        if effect_file and effect_file.exists():
            effect_img = nib.load(str(effect_file))
            effect_data = effect_img.get_fdata()
            
            # Get effect values within cluster
            effect_vals = effect_data[cluster_mask]
            if len(effect_vals) > 0:
                mean_es = np.mean(effect_vals)
        
        return peak_z, peak_mni, mean_es
        
    except Exception as e:
        print(f"    Error calculating stats for cluster {cluster_id}: {e}")
        return None, None, None

# Function to extract cluster information with statistics
def parse_cluster_file(filepath_obj):
    """Extract top 3 brain regions and their coverage percentages for each cluster"""
    with open(str(filepath_obj), 'r') as f:
        content = f.read()
    
    clusters = []
    
    # Find cluster and effect files
    cluster_file, effect_file, z_file = find_cluster_and_effect_files(filepath_obj)
    
    # Find all cluster sections
    cluster_sections = re.findall(
        r'\+\+ Processing unique value of (\d+)(.*?)'
        r'Intersection of ROI.*?\(sb0\):(.*?)'
        r'(?:\+\+ =|$)',
        content,
        re.DOTALL
    )
    
    for cluster_num, pre_intersection_text, intersection_text in cluster_sections:
        # Extract cluster size
        cluster_size = None
        size_match = re.search(r'(\d+)\s+voxels in ROI', pre_intersection_text)
        if size_match:
            cluster_size = int(size_match.group(1))
        
        # Calculate cluster statistics
        peak_z, peak_mni, mean_es = calculate_cluster_stats(
            cluster_file, effect_file, z_file, cluster_num
        )
        
        # Extract overlap percentages and region names
        overlaps = re.findall(
            r'([\d.]+)\s*%\s*overlap with\s+([^,]+),\s+code\s+(\d+)',
            intersection_text
        )
        
        # Take top 3
        top_regions = []
        for percentage, region, code in overlaps[:3]:
            top_regions.append({
                'region': region.strip(),
                'percentage': float(percentage),
                'code': code
            })
        
        if top_regions:  # Only add if there are regions
            clusters.append({
                'cluster_id': cluster_num,
                'cluster_size': cluster_size,
                'peak_z': peak_z,
                'peak_mni': peak_mni,
                'mean_es': mean_es,
                'regions': top_regions
            })
    
    return clusters

# Find all Brainnetome report files using glob
file_pattern = "*/atlas_coverage_reports/*Brainnetome_1.0_whereami_report.txt"
report_files_paths = sorted(results_dir.glob(file_pattern))

print(f"Found {len(report_files_paths)} files to process")
print(f"Results directory: {results_dir}\n")

# Parse all files
all_data = []
for filepath_obj in report_files_paths:
    filename = filepath_obj.name
    parent_dirname = filepath_obj.parent.parent.name
    full_path_str = str(filepath_obj)
    
    # Process files that contain 'qc_final_covariates' OR are in '_covar' folders
    if 'qc_final_covariates' not in full_path_str and not parent_dirname.endswith('_covar'):
        print(f"⊘ Skipping (not qc_final_covariates or _covar): {filename}")
        continue
    
    # Skip chi-square files
    if '_chisq_' in filename or 'chisq' in filename:
        print(f"⊘ Skipping (chi-square): {filename}")
        continue
    
    # Skip mod_view_vs_0 files (without Hi or Lo)
    if "mod_view_vs_0" in parent_dirname and "Hi" not in parent_dirname and "Lo" not in parent_dirname:
        print(f"⊘ Skipping (view_vs_0): {filename}")
        continue
    
    try:
        model, second_level, first_level = parse_filename(filepath_obj)
        
        # Skip files that don't match a known session pattern
        if second_level is None:
            print(f"⊘ Skipping (unknown session): {filename}")
            continue
            
        print(f"Processing: {filename}")
        clusters = parse_cluster_file(filepath_obj)
        
        if clusters:  # Only add if there are clusters
            all_data.append({
                'filepath': str(filepath_obj),
                'model': model,
                'second_level_contrast': second_level,
                'first_level_contrast': first_level,
                'clusters': clusters
            })
            print(f"✓ {filename} - {len(clusters)} cluster(s) - {first_level}")
        else:
            print(f"○ {filename} - No clusters found")
    except FileNotFoundError:
        print(f"✗ Not found: {filename}")
    except Exception as e:
        print(f"✗ Error: {filename} - {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f"Total files processed with clusters: {len(all_data)}")
print(f"{'='*80}\n")

if len(all_data) == 0:
    print("⚠ WARNING: No data to process! All files were either excluded or had no clusters.")
    print("   The document will not be created.")
else:
    # Split data into main effects and covariates
    main_effects_keywords = ['pre-surgery', '4 months post-surgery > pre-surgery', 
                              '12 months post-surgery > pre-surgery', '24 months post-surgery > pre-surgery',
                              'pre-surgery X TWL', '4 months post-surgery > pre-surgery X TWL',
                              '12 months post-surgery > pre-surgery X TWL', '24 months post-surgery > pre-surgery X TWL']
    
    covariate_keywords = ['Age baseline', 'BMI baseline', 'Handedness', 'Surgery type', 'Run']
    
    main_effects_data = []
    covariates_data = []
    
    for entry in all_data:
        second_level = entry['second_level_contrast'] or ''
        
        # Check if it's a main effect
        is_main_effect = any(keyword in second_level for keyword in main_effects_keywords)
        # Check if it's a covariate
        is_covariate = any(keyword in second_level for keyword in covariate_keywords)
        
        if is_main_effect:
            main_effects_data.append(entry)
        elif is_covariate:
            covariates_data.append(entry)
    
    print(f"Main effects entries: {len(main_effects_data)}")
    print(f"Covariates entries: {len(covariates_data)}")
    
    # Define sorting orders
    def get_sort_key(entry):
        model_order = 0 if entry['model'] == "session + covariates" else 1
        
        second_level = entry['second_level_contrast'] or ''
        second_level_order_map = {
            'pre-surgery': 0,
            '4 months post-surgery > pre-surgery': 1,
            '12 months post-surgery > pre-surgery': 2,
            '24 months post-surgery > pre-surgery': 3,
            'BMI baseline': 4,
            'Age baseline': 5,
            'Surgery type (RYGB > SG) at 4 months': 6,
            'Surgery type (BPD > SG) at 12 months': 7,
            'Run 2 > Run 1': 8,
            'Run 3 > Run 1': 9,
            'Handedness (left > right)': 10,
            'pre-surgery X TWL at 24 months': 11,
            '4 months post-surgery > pre-surgery X TWL at 24 months': 12,
            '12 months post-surgery > pre-surgery X TWL at 24 months': 13,
            '24 months post-surgery > pre-surgery X TWL at 24 months': 14,
            'BMI baseline X TWL at 24 months': 15,
            'Age baseline X TWL at 24 months': 16,
        }
        second_level_order = second_level_order_map.get(second_level, 999)
        
        first_level = entry['first_level_contrast'] or ''
        first_level_order_map = {
            'Valuating high calorie > low calorie': 0,
            'Viewing high calorie > low calorie': 1,
            'Valuating high calorie': 2,
            'Valuating low calorie': 3,
            'Valuating (all conditions)': 4,
        }
        first_level_order = first_level_order_map.get(first_level, 999)
        
        min_cluster_id = min(int(c['cluster_id']) for c in entry['clusters']) if entry['clusters'] else 999
        
        return (model_order, second_level_order, first_level_order, min_cluster_id)
    
    # Function to create a Word document
    def create_word_document(data, title_suffix, filename_suffix):
        if len(data) == 0:
            print(f"⚠ No data for {title_suffix}")
            return
        
        # Sort data hierarchically
        data.sort(key=get_sort_key)

        # Create Word document
        doc = Document()
        
        # Set narrow margins (0.5 inches all around)
        sections = doc.sections
        for section in sections:
            section.top_margin = Inches(0.5)
            section.bottom_margin = Inches(0.5)
            section.left_margin = Inches(0.5)
            section.right_margin = Inches(0.5)

        # Add title
        title = doc.add_heading(f'Brain Atlas Coverage Report - Brainnetome 1.0 (QC Final + Covariates) - {title_suffix}', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add subtitle
        subtitle = doc.add_paragraph('Cluster Analysis Results')
        subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
        subtitle.runs[0].font.size = Pt(12)
        subtitle.runs[0].italic = True

        doc.add_paragraph()

        # Create table with 10 columns
        table = doc.add_table(rows=1, cols=10)
        table.style = 'Table Grid'

        # Header row
        header_cells = table.rows[0].cells
        header_cells[0].text = 'Model'
        header_cells[1].text = 'Group Level Contrast'
        header_cells[2].text = 'Subject Level Contrast'
        header_cells[3].text = 'Cluster ID'
        header_cells[4].text = 'Cluster Size'
        header_cells[5].text = 'Peak Z'
        header_cells[6].text = 'Peak MNI'
        header_cells[7].text = 'Mean ES (%)'
        header_cells[8].text = 'Brainnetome label'
        header_cells[9].text = 'Coverage %'

        # Make header bold and set thick borders
        for cell in header_cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.bold = True
                    run.font.size = Pt(7)
            set_cell_border(
                cell,
                top={"sz": 12, "val": "single", "color": "#000000"},
                bottom={"sz": 12, "val": "single", "color": "#000000"},
                start={"sz": 12, "val": "single", "color": "#000000"},
                end={"sz": 12, "val": "single", "color": "#000000"},
            )
        
        # Make header row repeat on every page
        set_repeat_table_header(table.rows[0])

        # Add data rows with smart borders
        prev_model = None
        prev_second_level = None
        prev_first_level = None
        row_count = 0

        for entry in data:
            current_model = entry['model']
            current_second_level = entry['second_level_contrast']
            current_first_level = entry['first_level_contrast']
            
            for cluster in entry['clusters']:
                for i, region_info in enumerate(cluster['regions']):
                    row_cells = table.add_row().cells
                    row_count += 1
                    
                    # Determine if we need thick borders
                    is_new_model = current_model != prev_model
                    is_new_second_level = current_second_level != prev_second_level
                    is_new_first_level = current_first_level != prev_first_level and not is_new_second_level
                    
                    # Only fill model/contrast/cluster info when they change
                    if i == 0:
                        if is_new_model or is_new_second_level or is_new_first_level:
                            row_cells[0].text = entry['model'] if (is_new_model or is_new_second_level) else ''
                            row_cells[1].text = (entry['second_level_contrast'] or 'N/A') if (is_new_model or is_new_second_level) else ''
                            row_cells[2].text = (entry['first_level_contrast'] or 'N/A') if (is_new_model or is_new_second_level or is_new_first_level) else ''
                        else:
                            row_cells[0].text = ''
                            row_cells[1].text = ''
                            row_cells[2].text = ''
                        row_cells[3].text = cluster['cluster_id']
                        row_cells[4].text = str(cluster['cluster_size']) if cluster['cluster_size'] else 'N/A'
                        row_cells[5].text = f"{cluster['peak_z']:.2f}" if cluster['peak_z'] is not None else 'N/A'
                        row_cells[6].text = cluster['peak_mni'] if cluster['peak_mni'] else 'N/A'
                        row_cells[7].text = f"{cluster['mean_es']:.2f}" if cluster['mean_es'] is not None else 'N/A'
                    else:
                        for j in range(8):
                            row_cells[j].text = ''
                    
                    row_cells[8].text = region_info['region']
                    row_cells[9].text = f"{region_info['percentage']:.1f}"
                    
                    # Formatting
                    for cell in row_cells:
                        for paragraph in cell.paragraphs:
                            for run in paragraph.runs:
                                run.font.size = Pt(7)
                    
                    # Set borders
                    for j, cell in enumerate(row_cells):
                        border_config = {
                            "start": {"sz": 6, "val": "single", "color": "#000000"},
                            "end": {"sz": 6, "val": "single", "color": "#000000"},
                        }
                        
                        if is_new_second_level and i == 0:
                            border_config["top"] = {"sz": 12, "val": "single", "color": "#000000"}
                        elif is_new_first_level and i == 0 and j >= 2:
                            border_config["top"] = {"sz": 8, "val": "single", "color": "#000000"}
                        
                        set_cell_border(cell, **border_config)
                    
                    # Update tracking variables
                    if i == 0:
                        prev_model = current_model
                        prev_second_level = current_second_level
                        prev_first_level = current_first_level

        # Adjust column widths to fit in 7.5 inches total
        table.columns[0].width = Inches(0.9)
        table.columns[1].width = Inches(1.2)
        table.columns[2].width = Inches(1.1)
        table.columns[3].width = Inches(0.4)
        table.columns[4].width = Inches(0.5)
        table.columns[5].width = Inches(0.4)
        table.columns[6].width = Inches(0.85)
        table.columns[7].width = Inches(0.55)
        table.columns[8].width = Inches(1.2)
        table.columns[9].width = Inches(0.45)

        # Save document
        output_path = os.path.join(str(results_dir), f"Brainnetome_Cluster_Coverage_Report_QC_Final_Covariates_{filename_suffix}.docx")
        doc.save(output_path)

        print(f"✓ {title_suffix} document created successfully!")
        print(f"  Location: {output_path}")
        print(f"  Total rows: {row_count} (plus header)")
        print(f"  Unique combinations: {len(data)}")
        print()
    
    # Create both documents
    create_word_document(main_effects_data, "Main Effects", "MainEffects")
    create_word_document(covariates_data, "Covariates", "Covariates")
    
    print(f"{'='*80}")
    print(f"Both documents created successfully!")
    print(f"{'='*80}")

Found 208 files to process
Results directory: /home/gagpat01/Documents/GutBrain/BIDS_results/3dLmer/results

⊘ Skipping (not qc_final_covariates or _covar): sess1_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_Brainnetome_1.0_whereami_report.txt
⊘ Skipping (not qc_final_covariates or _covar): sess2_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_Brainnetome_1.0_whereami_report.txt
⊘ Skipping (not qc_final_covariates or _covar): sess2vs1_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_Brainnetome_1.0_whereami_report.txt
⊘ Skipping (not qc_final_covariates or _covar): handedness_chisq_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_mr_th_1_ru_qc_final_Brainnetome_1.0_whereami_report.txt
⊘ Skipping (not qc_final_covariates or _covar): sess2_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_mr_th_1_ru_qc_final_Brainnetome_1.0_whereami_report.txt
⊘ Skipping (not qc_final_co